In [ ]:
#%%apyter init
from appyter import magic
magic.init(lambda _=globals: _())

In [ ]:
%%appyter hide_code

{% do SectionField(
    name='gene_input', 
    title = '1. Select an input gene', 
    subtitle = 'Enter a human gene of interest'
) %}

{% do SectionField(
    name='method_input', 
    title = '2. Select a drug ranking method', 
    subtitle = "Select a ranking method by which to identify top up- and down-regulating drugs. Options are to rank drugs by the differential expression (DE) p-value of the target (default) or by the rank of the target in each perturbation's DE signature (Target Rank), where ranking is determined by the adjusted p-value of differential expression. This method prioritizes drug specificity over magnitude of regulation."
) %}

In [ ]:
%%appyter hide_code

{% set input_gene = AutocompleteField(
    name = 'input_gene',
    label = 'Query Gene',
    default = 'C9ORF72',
    description = 'Enter the gene symbol of interest.',
    file_path = 'https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/all_genes.json',
    section='gene_input'
)%}

{% set ranking_method = ChoiceField(
    name = 'ranking_method',
    label = 'Ranking Method',
    default = 'Differential Expression P-Value',
    description = "Rank drugs by the differential expression (DE) p-value of the target or the rank of the target in each perturbation's DE signature. Ranking is determined by the adjusted p-value of differential expression.",
    choices = [
        'Differential Expression P-Value',
        'Target Rank'
    ],
    section='method_input'
)%}

In [ ]:
%%appyter code_exec
query_gene = "{{ input_gene.value.upper() }}"

{%- if ranking_method.raw_value == 'Differential Expression P-Value' %}
ranking_method = 'pval'
{%- else %}
ranking_method = 'target_rank'
{%- endif %}

# Drug Gene Budger 2

This notebook takes a gene as input and identifies drugs that maximally up and down regulate the gene's expression in a collection of chemical perturbation datasets.

- Ginkgo GDPx1 and GPDx2: Limma-Voom based differential gene expression results for 1,354 drugs.
- Novartis DRUG-seq: Differential: Limma-Trend based differential gene expression results for 4,343 drugs. 
- LINCS L1000 Chemical Perturbations: Limma-Voom based differential gene expression results for a subset of 4,091 drugs from the LINCS L1000 Chemical Perturbation dataset. 

The Ginkgo dataset includes 4 primary cell types (epithelial melanocytes, smooth aortic muscle cells, skeletal muscle myoblasts and dermal fibroblasts) and one cell line (A549 lung carcinoma cell line). Previous analysis showed distinct transcriptional responses by cell type, so the drug rankings for the Ginkgo dataset are separated by cell type.

In [ ]:
## General
import pandas as pd
import numpy as np
import re
import warnings

## HTTP Requests
import requests

## Tables
from IPython.display import display, display_markdown, HTML

## UpSet Plot
from upsetplot import from_contents, plot
from matplotlib import pyplot

## Venn Diagram
from matplotlib_venn import venn3

## Volcano Plot
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool, LinearColorMapper
from bokeh.palettes import RdBu
from bokeh.io import output_notebook

In [ ]:
# Storage url for Ginkgo and Novartis DE files
ginkgo_URL = 'https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/ginkgo_de'
novartis_URL = 'https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/novartis_de'
lincs_URL = 'https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/lincs_de'
# silence warnings
warnings.filterwarnings('ignore')

In [ ]:
in_ginkgo = in_novartis = in_lincs = True

In [ ]:
# get Ginkgo DE results for gene
gene_file = f'{query_gene}.f'
try:
    ginkgo_de = pd.read_feather(f'{ginkgo_URL}/{gene_file}')
    ginkgo_cell_types = list(set(p.split('-')[0] for p in ginkgo_de.Perturbation))
except:
    in_ginkgo=False
    print('Gene not in Ginkgo dataset')
    

In [ ]:
def prepare_ginkgo_data(df, cell_types):
    '''Create a results dictionary where each cell type
    in the Ginkgo dataset is a key and the value is the DE data
    for the query gene for that cell type.
    '''
    # get perturbations with given cell type
    cell_type_results = {}
    for k in cell_types:
        subset = df[df['Perturbation'].str.contains(k)]
        subset['log10adj.P.Val'] = subset['adj.P.Val'].replace(0,1e-323).map(np.log10)*-1
        cell_type_results[k] = subset
    return cell_type_results
    

In [ ]:
if in_ginkgo:
    ginkgo_gene_expr_dict = prepare_ginkgo_data(ginkgo_de, ginkgo_cell_types)

In [ ]:
# get LINCS DE results for gene
try:
    lincs_de = pd.read_feather(f'{lincs_URL}/{gene_file}')
     # format p-values
    lincs_de['log10adj.P.Val'] = lincs_de['adj.P.Val'].replace(0,1e-323).map(np.log10)*-1
    lincs_ko_perturbs = pd.read_csv('https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/lincs_ko_perturbs.txt', sep='\t')
    lincs_de = lincs_de[~lincs_de['Drug'].isin(lincs_ko_perturbs.cmap_name.to_list())]
except:
    print('Gene not in LINCS L1000 dataset')
    in_lincs=False

In [ ]:
# get Novartis DE results for gene
try:
    novartis_de = pd.read_feather(f'{novartis_URL}/{gene_file}').set_index('index')
     # format p-values
    novartis_de['log10adj.P.Val'] = novartis_de['P.Adj'].replace(0,1e-323).map(np.log10)*-1
    # rename logFC column for concordance with Ginkgo columns
    novartis_de.rename(columns={'LogFC':'logFC'}, inplace=True)
except:
    print('Gene not in Novartis DRUG-seq dataset')
    in_novartis=False

In [ ]:
if in_lincs + in_novartis + in_ginkgo < 2:
    print(f"LINCS: {in_lincs}")
    print(f"Novartis: {in_novartis}")
    print(f"Ginkgo: {in_ginkgo}")
    raise Exception("Execution stopped, gene not found in at least 2 datasets")

## Query Gene

In [ ]:
display_markdown(f"This notebook shows results for the input gene **{query_gene}**", raw=True)
display_markdown(f"Drugs that up and down regulate **{query_gene}** are ranked by method **{ranking_method}**", raw=True)

## Rank Tables

Within each dataset drugs are ranked by either:

1. The statistical significance of the regulatory relationship. The pipeline uses the adjusted p-value from the differential expression results.

or

2. The rank of the query gene relative to all other targets. Rank is determined by differential expression adjusted p-value. A ranking of 1 indicates that the gene has the most significant adjusted p-value for differential expression compared to all other genes for that perturbation. 

When a dataset contains multiple perturbations for the same drug (i.e. a cell exposed to the drug at different doses), p-values and normalized ranks are averaged across doses to get a single ranking for the drug. 

The rankings are done separately for up-regulated and down-regulated genes.

In [ ]:
def get_rankings(data:pd.DataFrame, source:str, cell_type:str, direction:str, option:str):
    '''
    Given a dataframe of logFC and p-values for a gene of interest across perturbations, 
    rank the drugs by how the induce or repress the gene. 

    Returns a tuple of 1) drug ranks averaged across drug dosages and 2) full
    perturbation ranks. 
    '''
    ranked_data = data.copy()
    
    if (source == 'Ginkgo') & (cell_type=='A549'):
        ranked_data.loc[ranked_data['Drug']=='Brefeldin A from Penicillium brefeldianum', 'Drug'] = 'Brefeldin A'
    elif (source == 'Ginkgo') & (cell_type != 'A549'):
        ranked_data.loc[ranked_data['Drug']=='Brefeldin-A', 'Drug'] = 'Brefeldin A'
    elif source == 'Novartis':
        ranked_data.loc[ranked_data['Drug']=='Trichostatin A (racemate)', 'Drug'] = 'Trichostatin A'
        ranked_data.rename(columns={'P.Adj':'adj.P.Val'}, inplace=True)
    if option == 'pval':
        # average rank across all drug dosages
        drug_mean_ranks = ranked_data.loc[:,['Drug','logFC','log10adj.P.Val']].groupby('Drug')[['logFC','log10adj.P.Val']].mean().sort_values('log10adj.P.Val', ascending=False)
        # filter for up or down regulation
        if direction == 'up':
            drug_mean_ranks = drug_mean_ranks[drug_mean_ranks['logFC'] > 0]
        elif direction == 'down':
            drug_mean_ranks = drug_mean_ranks[drug_mean_ranks['logFC'] < 0]
        drug_mean_ranks.rename(columns={'logFC':'Avg logFC', 'log10adj.P.Val':'Avg -log10(Adj.PVal)'}, inplace=True)
    elif option == 'target_rank':
        if direction == 'up':
            ranked_data = ranked_data[ranked_data['GeneDir'] == 'Up']
        elif direction == 'down':
            ranked_data = ranked_data[ranked_data['GeneDir'] == 'Dn']
        drug_mean_ranks = ranked_data.loc[:,['Drug','logFC','adj.P.Val','Rank','PctRank']].groupby('Drug')[['logFC','adj.P.Val','Rank','PctRank']].mean().sort_values('PctRank', ascending=True)
    return drug_mean_ranks, ranked_data

def get_top(rank_results:pd.DataFrame, n=50):
    '''
    Given the drug_mean_ranks result from get_rankings, extract the names of the drugs
    that most down- or up-regulate the gene of interest (top N).

    If there are less drugs than N, will return all results.
    '''
    top = {d.casefold() for d in set(rank_results.head(n).index)}
    return top

# create download link for table results
def download_link(df, fname, link_header='Download full results'):
    if df.shape[0] == 0: return ''
    csv = df.to_csv(fname, sep='\t', index=True)
    link = f'<div>{link_header}: <a href="{fname}" target=_blank>{fname}</a></div>'
    return link

### Ginkgo

Drug rankings for the Ginkgo dataset. Top 20 by the chosen ranking method are shown, and the full results are available for download. 

In [ ]:
top_n = 20

In [ ]:
ginkgo_drugs_up = {}
ginkgo_drugs_down = {}
for cell_type, exprdf in ginkgo_gene_expr_dict.items():
    # rank by level of up-regulation
    mean_ranks, full_ranks = get_rankings(exprdf, 'Ginkgo', cell_type, 'up', ranking_method)
    ginkgo_drugs_up[cell_type] = (mean_ranks, full_ranks)
    display_markdown(f'**Top {top_n} up-regulators for {cell_type}**', raw=True)
    display(mean_ranks.head(top_n))
    display(HTML(download_link(mean_ranks, f"ginkgo_drug_ranks_{query_gene}_UpReg_{cell_type}.tsv", 'Download results averaged across drug dosages')))
    display(HTML(download_link(full_ranks, f"ginkgo_drug_ranks_{query_gene}_full_UpRg_{cell_type}.tsv", 'Download results for all perturbations')))
    # rank by level of down-regulation
    mean_ranks, full_ranks = get_rankings(exprdf, 'Ginkgo', cell_type, 'down', ranking_method)
    ginkgo_drugs_down[cell_type] = (mean_ranks, full_ranks)
    display_markdown(f'**Top {top_n} down-regulators for {cell_type}**', raw=True)
    display(mean_ranks.head(top_n))
    display(HTML(download_link(mean_ranks, f"ginkgo_drug_ranks_{query_gene}_DnReg_{cell_type}.tsv", 'Download results averaged across drug dosages')))
    display(HTML(download_link(full_ranks, f"ginkgo_drug_ranks_{query_gene}_full_DnRg_{cell_type}.tsv", 'Download results for all perturbations')))


### L1000

Drug rankings for the LINCS L1000 dataset. Top 20 by the chosen ranking method are shown, and the full results are available for download. 

In [ ]:
lincs_drugs_up = get_rankings(lincs_de, 'LINCS', '', 'up', ranking_method)
lincs_drugs_down = get_rankings(lincs_de, 'LINCS', '', 'down', ranking_method)
display_markdown(f'**Top {top_n} up-regulators in L1000**', raw=True)
display(lincs_drugs_up[0].head(top_n))
display(HTML(download_link(lincs_drugs_up[0], f"l1000_drug_ranks_{query_gene}_UpReg.tsv", 'Download results averaged across drug dosages')))
display(HTML(download_link(lincs_drugs_up[1], f"l1000_drug_ranks_{query_gene}_full_UpReg.tsv", 'Download results for all perturbations')))
display_markdown(f'**Top {top_n} down-regulators in L1000**', raw=True)
display(lincs_drugs_down[0].head(top_n))
display(HTML(download_link(lincs_drugs_down[0], f"l1000_drug_ranks_{query_gene}_DnReg.tsv", 'Download results averaged across drug dosages')))
display(HTML(download_link(lincs_drugs_down[1], f"l1000_drug_ranks_{query_gene}_full_DnReg.tsv", 'Download results for all perturbations')))

### Novartis DRUG-seq

Drug rankings for the Novartis DRUG-seq dataset. Top 20 by the chosen ranking method are shown, and the full results are available for download. 

In [ ]:
novartis_drugs_up = get_rankings(novartis_de, 'Novartis', '', 'up', ranking_method)
novartis_drugs_down = get_rankings(novartis_de, 'Novartis', '', 'down', ranking_method)

display_markdown(f'**Top {top_n} up-regulators in Novartis DRUG-seq**', raw=True)
display(novartis_drugs_up[0].head(top_n))
display(HTML(download_link(novartis_drugs_up[0], f'novartis_drug_ranks_{query_gene}_UpReg.tsv', 'Download results averaged across drug dosages')))
display(HTML(download_link(novartis_drugs_up[1], f'novartis_drug_ranks_{query_gene}_full_UpReg.tsv', 'Download results for all perturbations')))
display_markdown(f'**Top {top_n} down-regulators in Novartis DRUG-seq**', raw=True)
display(novartis_drugs_down[0].head(top_n))
display(HTML(download_link(novartis_drugs_down[0], f'novartis_drug_ranks_{query_gene}_DnReg.tsv', 'Download results averaged across drug dosages')))
display(HTML(download_link(novartis_drugs_down[1], f'novartis_drug_ranks_{query_gene}_full_DnReg.tsv', 'Download results for all perturbations')))

In [ ]:
top_up = {}
top_down = {}
# get results from Ginkgo
for cell_type in ginkgo_drugs_down.keys():
    top_up[f'ginkgo_{cell_type}'] = get_top(ginkgo_drugs_up[cell_type][0], n=50)
    top_down[f'ginkgo_{cell_type}'] = get_top(ginkgo_drugs_down[cell_type][0], n=50)
# get results from L1000
top_up['lincs_l1000'] = {drug.casefold() for drug in set(lincs_drugs_up[0].index)}
top_down['lincs_l1000'] = {drug.casefold() for drug in set(lincs_drugs_down[0].index)}
# get results from novartis
top_up['novartis'] = get_top(novartis_drugs_up[0], n=50)
top_down['novartis'] = get_top(novartis_drugs_down[0], n=50)

## UpSet Plot

The UpSet plots show the overlap among top up-regulating or down-regulating drugs in each dataset. If there were more than 50 significant regulators in a dataset for a given input gene, the input was restricted to the top 50 regulators.

In [ ]:
# Saving Figures
def save_figure(plot_name, **kwargs):
    import io
    mem = io.BytesIO()
    pyplot.savefig(mem, bbox_inches='tight')
    with open(plot_name, 'wb') as fw:
        fw.write(mem.getbuffer())

In [ ]:
def create_upset(top_sets: dict, ranking_method:str):
    rename_keys = {
            'ginkgo_A549': 'ginkgo_A549',
            'lincs_l1000': 'lincs_l1000',
            'novartis': 'novartis',
            'ginkgo_human_epithelial_melanocytes': 'ginkgo_melanocytes',
            'ginkgo_human_dermal_fibroblast': 'ginkgo_fibroblasts',
            'ginkgo_human_aortic_smooth_muscle_cells': 'ginkgo_muscle_cells',
            'ginkgo_human_skeletal_muscle_myoblasts': 'ginkgo_myoblasts'
    }
    top_sets = {rename_keys[k]:v for k,v in top_sets.items()}
    upset_data = from_contents(top_sets)
    plot(upset_data, orientation = 'horizontal', show_counts = True, element_size = 30)
    pyplot.show()

In [ ]:
display_markdown(f"**Overlap among top up regulators of {query_gene}**", raw=True)
create_upset(top_up, ranking_method)

In [ ]:
display_markdown(f"**Overlap among top down regulators of {query_gene}**", raw=True)
create_upset(top_down, ranking_method)

In [ ]:
def get_overlapping_sets(top_sets:dict, to_file:str):
    '''
    Given the dictionary of sets used to created the UpSet plot,
    return the contents of the overlapping sets. 
    '''
    # convert to multi-index dataframe
    set_df = from_contents(top_sets)
    multi_index_df = pd.DataFrame(columns=list(top_sets.keys()))
    for colname in multi_index_df.columns:
        multi_index_df[colname] = set_df.index.get_level_values(colname).to_list()
    # only keep unique sets of intersection contributors
    multi_index_df.drop_duplicates(inplace=True)
    # sort multi-index for efficient indexing
    set_df = set_df.sort_index()
    # extract drug intersection for each group
    overlapping_sets = pd.DataFrame(columns=['Members', 'Overlap', 'Length'])
    for idx in range(multi_index_df.shape[0]):
        ixn_drugs = set_df.loc[tuple(multi_index_df.iloc[idx])].id.to_list()
        # get group members
        ixn_name = multi_index_df.iloc[idx][multi_index_df.iloc[idx]].index.to_list()
        ixn_name_joined = '-'.join(ixn_name)
        # append results
        overlapping_sets = pd.concat([overlapping_sets, pd.DataFrame({'Members':ixn_name_joined, 'Overlap':[ixn_drugs], 'Length':len(ixn_drugs), 'N Datasets':len(ixn_name)})])
        
    
    overlapping_sets = overlapping_sets.sort_values('N Datasets', ascending=False)
    return overlapping_sets


## Consensus Regulator Tables

Below are tabular representations of the UpSet plots.

In [ ]:
overlap_down = get_overlapping_sets(top_down, 'overlap_down')
overlap_up = get_overlapping_sets(top_up, 'overlap_up')
display_markdown("**Down-regulating drug overlap**", raw=True)
display(overlap_down)
display(HTML(download_link(overlap_down, f'overlapping_drugs_{query_gene}_DnReg.tsv')))
display_markdown("**Up-regulating drug overlap**", raw=True)
display(overlap_up)
display(HTML(download_link(overlap_up, f'overlapping_drugs_{query_gene}_UpReg.tsv')))


In [ ]:
def get_ranking_averages(overlapping_df, direction, ranking_method):
    '''
    Retrieve average target ranking across datasets for drugs in overlapping sets. 

    Returns dataframe with columns for:
    Drug
    Average Rank
    Number of datasets for which drug was a significant regulator of the query gene
    '''
    # extract up or down data
    if direction == 'down':
        data_dict ={
            'ginkgo_A549': ginkgo_drugs_down['A549'][1],
            'ginkgo_human_dermal_fibroblast': ginkgo_drugs_down['human_dermal_fibroblast'][1],
            'ginkgo_human_aortic_smooth_muscle_cells': ginkgo_drugs_down['human_aortic_smooth_muscle_cells'][1],
            'ginkgo_human_epithelial_melanocytes':ginkgo_drugs_down['human_epithelial_melanocytes'][1],
            'ginkgo_human_skeletal_muscle_myoblasts':ginkgo_drugs_down['human_skeletal_muscle_myoblasts'][1],
            'novartis': novartis_drugs_down[1],
            'lincs': lincs_drugs_down[1]
        }
    elif direction == 'up':
        data_dict ={
            'ginkgo_A549': ginkgo_drugs_up['A549'][1],
            'ginkgo_human_dermal_fibroblast': ginkgo_drugs_up['human_dermal_fibroblast'][1],
            'ginkgo_human_aortic_smooth_muscle_cells': ginkgo_drugs_up['human_aortic_smooth_muscle_cells'][1],
            'ginkgo_human_epithelial_melanocytes':ginkgo_drugs_up['human_epithelial_melanocytes'][1],
            'ginkgo_human_skeletal_muscle_myoblasts':ginkgo_drugs_up['human_skeletal_muscle_myoblasts'][1],
            'novartis': novartis_drugs_up[1],
            'lincs': lincs_drugs_up[1]
        }
    # get average, integrating across datasets
    average_rank_vals = {}
    average_pctrank_vals = {}
    average_logfc_vals = {}
    average_pvals = {}
    n_datasets = list()
    for _,row in overlapping_df.iterrows():
        n_datasets.extend([row['N Datasets']]*len(row['Overlap']))
        for d in row['Overlap']:
            n = 0
            runsum_rank = 0
            runsum_pctrank = 0
            runsum_logFC = 0
            runsum_pval = 0
            for k,df in data_dict.items():
                subset = df[df['Drug'].str.lower() == d.lower()]
                n = n + subset.shape[0]
                runsum_rank = runsum_rank + subset.Rank.sum()
                runsum_pctrank = runsum_pctrank + subset.PctRank.sum()
                runsum_logFC = runsum_logFC + subset.logFC.sum()
                runsum_pval = runsum_pval + subset['adj.P.Val'].sum()
            average_rank_vals[d] = round(runsum_rank / n,3)
            average_pctrank_vals[d] = round(runsum_pctrank/n, 3)
            average_logfc_vals[d] = round(runsum_logFC/n, 3)
            average_pvals[d] = round(runsum_pval/n,3)
    # create results dataframe
    res_df = pd.DataFrame({
        'Drug': list(average_rank_vals.keys()),
        'Avg LogFC': list(average_logfc_vals.values()),
        'Avg Adj.P.Val': list(average_pvals.values()),
        'Avg Rank': list(average_rank_vals.values()),
        'Avg PctRank': list(average_pctrank_vals.values())
    })
    res_df['N Datasets'] = n_datasets
    if ranking_method == 'target_rank':
        # sort based on N datasets and average percentile rank
        res_df = res_df.sort_values(['N Datasets','Avg PctRank'], ascending=[False,True])
    else:
        # sort based on N datasets and average adjusted p-value
        res_df = res_df.sort_values(['N Datasets','Avg Adj.P.Val'], ascending=[False,True])
    return res_df
        
overlapping_up_TargetRank = get_ranking_averages(overlap_up, 'up', ranking_method)
overlapping_down_TargetRank = get_ranking_averages(overlap_down, 'down', ranking_method)

The tables below show average logFC, adjusted p-value, raw rank and normalized rank values across datasets for drugs that were found to be significant regulators in more than one dataset.

In [ ]:
display_markdown("**Averages across datasets: Up-regulating drugs**", raw=True)
display(overlapping_up_TargetRank.head(n=top_n))
display(HTML(download_link(overlapping_up_TargetRank, f'overlapping_drugs_averages_{query_gene}_UpReg.tsv')))
display_markdown("**Averages across datasets: Down-regulating drugs**", raw=True)
display(overlapping_down_TargetRank.head(n=top_n))
display(HTML(download_link(overlapping_down_TargetRank, f'overlapping_drugs_averages_{query_gene}_DnReg.tsv')))

## Venn Diagrams

The venn diagrams show the overlap among either up-regulating or down-regulating drugs across the three datasets Novartis DRUG-seq, LINCS L1000, and Ginkgo (all cell types grouped). 

In [ ]:
# combine top up and down drugs across Ginkgo cell types
all_ginkgo_up = set()
all_ginkgo_down = set()
for k,v in top_up.items():
    if re.search('ginkgo',k):
        all_ginkgo_up = all_ginkgo_up.union(v)
        all_ginkgo_down = all_ginkgo_down.union(top_down[k])

venn_up = {
        'novartis': top_up['novartis'],
        'lincs' : top_up['lincs_l1000'],
        'ginkgo' : all_ginkgo_up
}
venn_down = {
        'novartis': top_down['novartis'],
        'lincs' : top_down['lincs_l1000'],
        'ginkgo' : all_ginkgo_down
}

def print_overlap(venn_dict):
    gnl = venn_dict['ginkgo'].intersection(venn_dict['novartis']).intersection(venn_dict['lincs'])
    gn = venn_dict['ginkgo'].intersection(venn_dict['novartis']).difference(gnl)
    gl = venn_dict['ginkgo'].intersection(venn_dict['lincs']).difference(gnl)
    nl = venn_dict['novartis'].intersection(venn_dict['lincs']).difference(gnl)
    
    if len(gn) > 0:
        display_markdown(f'Novartis and Ginkgo: {[d for d in gn]}', raw=True)
    if len(nl) > 0:
        display_markdown(f'Novartis and LINCS L1000: {[d for d in nl]}', raw=True)
    if len(gl) > 0:
        display_markdown(f'LINCS L1000 and Ginkgo: {[d for d in gl]}', raw=True)
    if len(gnl) > 0:
        display_markdown(f'LINCS L1000, Ginkgo and Novartis: {[d for d in gnl]}', raw=True)

In [ ]:
display_markdown(f'Overlap of top {query_gene} up-regulating drugs across sources', raw=True)

venn3(subsets=(venn_up['novartis'], venn_up['lincs'], venn_up['ginkgo']),
            set_labels=('Novartis', 'LINCS L1000', 'Ginkgo'));
print_overlap(venn_up)

In [ ]:
display_markdown(f'Overlap of top {query_gene} down-regulating drugs across sources', raw=True)

venn3(subsets=(venn_down['novartis'], venn_down['lincs'], venn_down['ginkgo']),
            set_labels=('Novartis', 'LINCS L1000', 'Ginkgo'));
print_overlap(venn_down)

## Volcano Plots

The volcano plots show the strength and statistical significance of the drug perturbation for each signature in the dataset (drug and dose specific). Color of points indicate up (red) or down (blue) regulation. Hover over points in the volcano plot to see the label (with cell line, drug, and dose information), logFC, fold-change, log10-transformed p-value, raw rank and normalized rank. Tools to the right of the plot allow you to manipulate (pan, zoom) and download the figure. 

In [ ]:
output_notebook()

def create_bokeh_volcano_plot(expr_data:pd.DataFrame, gene_id:str, cell_type:str, source:str):
    '''
    Given the expression data for a given gene, create an interactive
    volcano plot that shows regulation of gene across all perturbations (drug, dosage, cell line).
    '''
    
    df = expr_data.copy()
    
    # clean columns
    df['FC'] = 2**df['logFC']
    if source == 'Ginkgo':
        df['Label'] = df['Perturbation']
    elif source == 'Novartis':
        df['Label'] = df['Perturbation'] + '_' + df['Drug']
    elif source == 'L1000':
        df['Label'] = df['Perturbation']

    # set plot source
    plot_source = ColumnDataSource(df.loc[:,['Label','logFC','FC','log10adj.P.Val', 'Rank', 'PctRank']])
    x,y='logFC','log10adj.P.Val'
    hover = HoverTool(tooltips=[("Label", "@Label"),
                            ("Log2(FC)", "@logFC"),
                            ("Fold Change", "@FC"),
                            ('-Log10(Adj. p-value)',"@{log10adj.P.Val}{0.00e}"),
                            ("Raw Rank", "@Rank"),
                            ("Normalized Rank", "@PctRank")])
        
    # define figure
    p = figure(
        title=f'{gene_id} Regulation in {source} {cell_type}',
        x_axis_label = 'Log2(Fold Change)',
        y_axis_label = '-Log10(Adj. p-value)',
        tools = 'pan,wheel_zoom,box_zoom,reset,save'
    )

    # color mapper
    color_mapper = LinearColorMapper(palette = RdBu[10],
                                     low = min(df['logFC']),
                                     high=max(df['logFC']))
    # plot
    p.scatter(x=x,
              y=y,
              size=8,
              source=plot_source,
              fill_alpha=0.6,
              color = {'field':'logFC','transform':color_mapper})
    p.add_tools(hover)
    show(p)

### Ginkgo

In [ ]:
for cell_type, expr_df in ginkgo_gene_expr_dict.items():
    cell_name = ' '.join(re.sub('human_','',cell_type).split('_'))
    create_bokeh_volcano_plot(expr_df, query_gene, cell_name, 'Ginkgo')

### L1000

In [ ]:
create_bokeh_volcano_plot(lincs_de, query_gene, '', 'L1000')

### Novartis DRUG-seq

In [ ]:
create_bokeh_volcano_plot(novartis_de, query_gene, '', 'Novartis')

## References

[1] Baugh, Lauren, Sébastien Vigneau, Srijani Sridhar, Sarah Boswell, George Pilitsis, John Bradley, Olga Allen, et al. 2025. “Mapping the Transcriptional Landscape of Drug Responses in Primary Human Cells Using High-Throughput DRUG-Seq.” bioRxiv. https://doi.org/10.1101/2025.06.03.657593.

[2] Datapoints, Ginkgo. n.d. “GDPx1.” Accessed September 5, 2025. https://huggingface.co/datasets/ginkgo-datapoints/GDPx1.

[3] Hadjikyriacou, Andrea, Chian Yang, Martin Henault, Robin Ge, Leandra Mansur, Alicia Lindeman, Carsten Russ, et al. 2025. “Novartis/DRUG-Seq U2OS MoABox Dataset.” Zenodo. https://doi.org/10.5281/ZENODO.14291446.

[4] Subramanian, Aravind, Rajiv Narayan, Steven M. Corsello, David D. Peck, Ted E. Natoli, Xiaodong Lu, Joshua Gould, et al. 2017. “A next Generation Connectivity Map: L1000 Platform and the First 1,000,000 Profiles.” Cell 171 (6): 1437-1452.e17.

[5] “LINCS L1000 Reverse Search.” n.d. Accessed September 5, 2025. https://lincs-reverse-search-dashboard.dev.maayanlab.cloud/.

[6] Wang, Zichen, Edward He, Kevin Sani, Kathleen M. Jagodnik, Moshe C. Silverstein, and Avi Ma’ayan. 2019. “Drug Gene Budger (DGB): An Application for Ranking Drugs to Modulate a Specific Gene Based on Transcriptomic Signatures.” Bioinformatics (Oxford, England) 35 (7): 1247–48.